# 🏥 Gender-Aware Early Disease Risk Prediction System
### Multi-Disease AI Health Engine · 9 Diseases · Personalized Risk Scoring
---
| | |
|---|---|
| **Diseases** | Breast, PCOS, Thyroid, Prostate, Heart, Diabetes, Kidney, Liver, Lung |
| **Algorithm** | Random Forest (Calibrated Probabilities) |
| **Gender-Aware** | ✅ Weighted risk prioritization per gender |
| **Output** | Per-disease risk % + Overall Health Score + Visual Report |

---
## 📋 Execution Order
Run cells **top to bottom** in order. Do NOT skip any cell.

## 📦 CELL 1 — Install Libraries & Setup

In [ ]:
# ══════════════════════════════════════════════════════════
# CELL 1 ▸ Install Dependencies & Import Libraries
# ══════════════════════════════════════════════════════════

!pip install scikit-learn imbalanced-learn openpyxl -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
import pickle
import os
import zipfile

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score, roc_auc_score, classification_report,
    confusion_matrix, roc_curve, ConfusionMatrixDisplay
)
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

try:
    from imblearn.over_sampling import SMOTE
    SMOTE_AVAILABLE = True
except ImportError:
    SMOTE_AVAILABLE = False
    print('ℹ️  imbalanced-learn not available — using class_weight="balanced" instead')

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'

os.makedirs('models', exist_ok=True)
os.makedirs('reports', exist_ok=True)

print('✅ All libraries imported successfully!')
print('📁 Created: models/  reports/')

## 📤 CELL 2 — Upload the 9 Dataset Files

In [ ]:
# ══════════════════════════════════════════════════════════
# CELL 2 ▸ Upload Datasets
# ══════════════════════════════════════════════════════════
# Upload all 9 files when the dialog appears.
# Required files:
#   breast_cancer_cleaned.csv      pcos_cleaned_dataset.csv
#   thyroid_cleaned_dataset.csv    prostate_cleaned_dataset.csv
#   heart_cleaned.xlsx         diabetes_cleaned.csv
#   kidney_cleaned_dataset.csv     liver_cleaned.csv
#   lung_cleaned.csv

from google.colab import files
uploaded = files.upload()

print(f'\n✅ Uploaded {len(uploaded)} file(s):')
for f in uploaded:
    size_kb = len(uploaded[f]) // 1024
    print(f'   • {f}  ({size_kb} KB)')

## ⚙️ CELL 3 — Dataset Configuration (Pre-filled — No Edits Needed)

In [ ]:
# ══════════════════════════════════════════════════════════
# CELL 3 ▸ Dataset Configuration
# ══════════════════════════════════════════════════════════
# ✅ All target columns are already mapped correctly from dataset analysis.
# ⚠️  DO NOT EDIT unless your filenames differ.

DISEASE_CONFIG = {
    #  key        file                            ftype   target              gender  display name
    'breast':   {'file':'breast_cancer_cleaned.csv',        'ftype':'csv',  'target':'diagnosis',     'gender':'female', 'display':'Breast Cancer'},
    'pcos':     {'file':'pcos_cleaned_dataset.csv',         'ftype':'csv',  'target':'PCOS (Y/N)',    'gender':'female', 'display':'PCOS'},
    'thyroid':  {'file':'thyroid_cleaned_dataset.csv',      'ftype':'csv',  'target':'Recurred_Yes',  'gender':'both',   'display':'Thyroid Disease'},
    'prostate': {'file':'prostate_cleaned_dataset.csv',     'ftype':'csv',  'target':'lpsa',          'gender':'male',   'display':'Prostate Cancer'},
    'heart':    {'file':'heart_cleaned.xlsx',           'ftype':'xlsx', 'target':'target',        'gender':'both',   'display':'Heart Disease'},
    'diabetes': {'file':'diabetes_cleaned.csv',             'ftype':'csv',  'target':'Outcome',       'gender':'both',   'display':'Diabetes'},
    'kidney':   {'file':'kidney_cleaned_dataset.csv',       'ftype':'csv',  'target':'Target',        'gender':'both',   'display':'Kidney Disease'},
    'liver':    {'file':'liver_cleaned.csv',                'ftype':'csv',  'target':'Dataset',       'gender':'both',   'display':'Liver Disease'},
    'lung':     {'file':'lung_cleaned.csv',                 'ftype':'csv',  'target':'Recovered',     'gender':'both',   'display':'Lung Cancer'},
}

# ── Gender-Aware Priority Weights ──────────────────────────
GENDER_WEIGHTS = {
    'female': {
        'breast':1.0, 'pcos':0.9,  'thyroid':0.9,
        'heart':0.7,  'liver':0.6,  'kidney':0.6,
        'diabetes':0.6, 'lung':0.5, 'prostate':0.0
    },
    'male': {
        'prostate':1.0, 'heart':0.9, 'liver':0.8,
        'lung':0.8,     'kidney':0.7, 'diabetes':0.7,
        'thyroid':0.5,  'breast':0.0, 'pcos':0.0
    }
}

# ── Risk Level Thresholds ──────────────────────────────────
def get_risk_level(score):
    if   score < 20: return ('LOW',      '🟢', '#2ecc71')
    elif score < 45: return ('MODERATE', '🟡', '#f39c12')
    elif score < 65: return ('HIGH',     '🔴', '#e74c3c')
    else:            return ('CRITICAL', '🚨', '#c0392b')

print('✅ Configuration loaded!')
print(f'   Diseases: {list(DISEASE_CONFIG.keys())}')
print(f'   Female-priority diseases: breast, pcos, thyroid')
print(f'   Male-priority diseases:   prostate, heart, liver, lung')

## 🔍 CELL 4 — Exploratory Data Analysis (EDA)

In [ ]:
# ══════════════════════════════════════════════════════════
# CELL 4 ▸ Load & EDA All 9 Datasets
# ══════════════════════════════════════════════════════════

raw_datasets = {}
eda_rows = []

for key, cfg in DISEASE_CONFIG.items():
    try:
        df = pd.read_excel(cfg['file']) if cfg['ftype']=='xlsx' else pd.read_csv(cfg['file'])
        raw_datasets[key] = df
        target = cfg['target']
        null_pct = round(df.isnull().sum().sum() / (df.shape[0]*df.shape[1])*100, 2)
        eda_rows.append({
            'Disease':      cfg['display'],
            'Rows':         df.shape[0],
            'Features':     df.shape[1]-1,
            'Null %':       null_pct,
            'Target Col':   target,
            'Gender Focus': cfg['gender']
        })
        print(f'✅ {cfg["display"]:20s} | rows={df.shape[0]:6d} | cols={df.shape[1]:3d} | nulls={null_pct}%')
    except FileNotFoundError:
        print(f'❌ {cfg["display"]:20s} | FILE NOT FOUND: {cfg["file"]}')

print(f'\n📊 Loaded {len(raw_datasets)}/9 datasets')

# ── Visual EDA: Class Distribution ────────────────────────
fig, axes = plt.subplots(3, 3, figsize=(15, 10))
fig.patch.set_facecolor('#0f1117')
axes = axes.flatten()

for idx, (key, df) in enumerate(raw_datasets.items()):
    ax = axes[idx]
    ax.set_facecolor('#1a1d2e')
    target = DISEASE_CONFIG[key]['target']
    try:
        counts = df[target].value_counts()
        colors = ['#e74c3c','#2ecc71'] if len(counts)==2 else plt.cm.Set2.colors[:len(counts)]
        ax.bar(range(len(counts)), counts.values, color=colors, edgecolor='none', width=0.55)
        ax.set_xticks(range(len(counts)))
        ax.set_xticklabels([str(v)[:8] for v in counts.index], color='white', fontsize=8)
        ax.set_title(DISEASE_CONFIG[key]['display'], color='white', fontweight='bold', fontsize=10)
        ax.set_ylabel('Count', color='#adb5bd', fontsize=8)
        ax.tick_params(colors='#adb5bd')
        ax.spines[:].set_visible(False)
        for i, v in enumerate(counts.values):
            ax.text(i, v+5, str(v), ha='center', color='white', fontsize=8)
    except:
        ax.set_visible(False)

plt.suptitle('📊 Class Distribution — All 9 Disease Datasets',
             color='white', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('reports/eda_class_distribution.png', bbox_inches='tight', facecolor='#0f1117', dpi=130)
plt.show()

print('\n📋 EDA Summary:')
print(pd.DataFrame(eda_rows).to_string(index=False))

## 🛠️ CELL 5 — Preprocessing Engine

In [ ]:
# ══════════════════════════════════════════════════════════
# CELL 5 ▸ Universal Preprocessing Engine
# ══════════════════════════════════════════════════════════
# Handles all 9 datasets automatically:
#   • Prostate  → binarize continuous lpsa at median
#   • Kidney    → binarize 5-class Target (0=healthy, 1-4=sick)
#   • Thyroid   → bool to int conversion
#   • PCOS      → force-coerce malformed numeric strings
#   • All       → encode categoricals, fill NaN with median

def preprocess_dataset(key, df):
    cfg    = DISEASE_CONFIG[key]
    target = cfg['target']
    df     = df.copy()

    # ── Step 1: Force all non-target columns to numeric ────
    for col in df.columns:
        if col == target:
            continue
        if df[col].dtype == 'bool':
            df[col] = df[col].astype(int)
        elif df[col].dtype == 'object':
            # Try numeric coercion first (handles '1.99.' typos)
            converted = pd.to_numeric(df[col], errors='coerce')
            if converted.notna().sum() >= 0.6 * len(df):
                df[col] = converted
            else:
                df[col] = LabelEncoder().fit_transform(df[col].astype(str))

    # ── Step 2: Binarize target column ────────────────────
    if key == 'prostate':
        # lpsa is continuous → binarize at median (high PSA = positive)
        median_val = df[target].median()
        df[target] = (df[target] >= median_val).astype(int)
        print(f'   ℹ️  Prostate: lpsa binarized at median={median_val:.3f}')

    elif key == 'kidney':
        # 5-class: 0=healthy, 1-4=disease stages → binary
        df[target] = (df[target] > 0).astype(int)
        print(f'   ℹ️  Kidney: 5-class target binarized (0=healthy, 1+=disease)')

    elif key == 'thyroid':
        # Boolean column
        df[target] = df[target].astype(int)

    elif df[target].dtype == 'bool':
        df[target] = df[target].astype(int)

    elif df[target].dtype == 'object':
        le = LabelEncoder()
        df[target] = le.fit_transform(df[target].astype(str))
    else:
        df[target] = df[target].astype(int)

    # ── Step 3: Drop ID-like columns ──────────────────────
    id_cols = [c for c in df.columns
               if c.lower().strip() in ['id','patientid','patient_id','sl. no','sl.no','index','serial no']]
    df.drop(columns=id_cols, inplace=True, errors='ignore')

    X = df.drop(columns=[target])
    y = df[target]
    return X, y


# ── Run preprocessing on all datasets ─────────────────────
processed_data = {}
print('🛠️  Preprocessing all 9 datasets...\n')

for key, df in raw_datasets.items():
    try:
        X, y = preprocess_dataset(key, df)
        processed_data[key] = (X, y)
        pos_pct  = round(y.mean()*100, 1)
        neg_pct  = round(100-pos_pct, 1)
        print(f'  ✅ {DISEASE_CONFIG[key]["display"]:20s} | features={X.shape[1]:3d} | '
              f'pos={pos_pct}% | neg={neg_pct}%')
    except Exception as e:
        import traceback
        print(f'  ❌ {DISEASE_CONFIG[key]["display"]:20s} | ERROR: {e}')
        traceback.print_exc()

print(f'\n✅ Preprocessing complete: {len(processed_data)}/9 datasets ready')

## 🤖 CELL 6 — Train All 9 Disease Models

In [ ]:
# ══════════════════════════════════════════════════════════
# CELL 6 ▸ Train All 9 Disease-Specific Models
# ══════════════════════════════════════════════════════════
# Architecture per model:
#   SimpleImputer (median) → StandardScaler → RandomForest
#   → CalibratedClassifierCV (Platt sigmoid scaling)
# Calibration ensures predict_proba returns meaningful
# probabilities (not just raw votes from RF)

trained_models   = {}
model_metrics    = []

def train_model(key, X, y):
    cfg = DISEASE_CONFIG[key]
    print(f'  ▸ Training: {cfg["display"]:20s}', end='  ')

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    # ── Optional SMOTE for class imbalance ────────────────
    if SMOTE_AVAILABLE:
        min_count = y_train.value_counts().min()
        majority_ratio = y_train.value_counts().max() / len(y_train)
        if min_count >= 6 and majority_ratio > 0.65:
            k = min(5, min_count - 1)
            sm = SMOTE(random_state=42, k_neighbors=k)
            try:
                X_train, y_train = sm.fit_resample(X_train, y_train)
            except Exception:
                pass

    # ── Build pipeline ────────────────────────────────────
    base_rf = RandomForestClassifier(
        n_estimators    = 300,
        max_depth       = 12,
        min_samples_leaf= 4,
        class_weight    = 'balanced',
        random_state    = 42,
        n_jobs          = -1
    )

    pipeline = Pipeline([
        ('imputer',    SimpleImputer(strategy='median')),
        ('scaler',     StandardScaler()),
        ('classifier', base_rf)
    ])

    # ── Calibrate probabilities ───────────────────────────
    cv_folds = min(3, y_train.value_counts().min())  # handle small datasets
    cv_folds = max(2, cv_folds)
    calibrated = CalibratedClassifierCV(pipeline, cv=cv_folds, method='sigmoid')
    calibrated.fit(X_train, y_train)

    # ── Evaluate ──────────────────────────────────────────
    y_pred = calibrated.predict(X_test)
    y_prob = calibrated.predict_proba(X_test)[:, 1]
    acc    = round(accuracy_score(y_test, y_pred)*100, 2)
    try:
        auc = round(roc_auc_score(y_test, y_prob)*100, 2)
    except:
        auc = 0.0

    print(f'Accuracy={acc}%  AUC={auc}%')

    # ── Save model to disk ────────────────────────────────
    model_path = f'models/{key}_model.pkl'
    with open(model_path, 'wb') as f:
        pickle.dump({'model': calibrated, 'features': list(X.columns), 'key': key}, f)

    return calibrated, list(X.columns), acc, auc, X_test, y_test, y_prob


print('🚀 Training all 9 disease models...\n')

for key, (X, y) in processed_data.items():
    model, features, acc, auc, X_test, y_test, y_prob = train_model(key, X, y)
    trained_models[key] = {
        'model':    model,
        'features': features,
        'X_test':   X_test,
        'y_test':   y_test,
        'y_prob':   y_prob
    }
    model_metrics.append({
        'Disease':      DISEASE_CONFIG[key]['display'],
        'Accuracy (%)': acc,
        'AUC-ROC (%)':  auc
    })

print(f'\n✅ All {len(trained_models)} models trained & saved to models/')
print('\n📋 Performance Summary:')
print(pd.DataFrame(model_metrics).to_string(index=False))

## 📊 CELL 7 — Model Performance Dashboard

In [ ]:
# ══════════════════════════════════════════════════════════
# CELL 7 ▸ Professional Performance Dashboard
# ══════════════════════════════════════════════════════════

perf_df = pd.DataFrame(model_metrics)
BG_DARK  = '#0f1117'
BG_CARD  = '#1a1d2e'
TXT      = 'white'
TXT_DIM  = '#adb5bd'

fig = plt.figure(figsize=(22, 14), facecolor=BG_DARK)
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.42, wspace=0.32)

# ── Accuracy ───────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
ax1.set_facecolor(BG_CARD)
colors = ['#2ecc71' if v>=80 else '#f39c12' if v>=65 else '#e74c3c'
          for v in perf_df['Accuracy (%)']]
bars = ax1.barh(perf_df['Disease'], perf_df['Accuracy (%)'],
                color=colors, edgecolor='none', height=0.6)
ax1.set_xlim(0, 112)
ax1.set_xlabel('Accuracy (%)', color=TXT_DIM)
ax1.set_title('Model Accuracy', color=TXT, fontweight='bold', pad=10)
ax1.tick_params(colors=TXT_DIM); ax1.spines[:].set_visible(False)
ax1.set_yticklabels(perf_df['Disease'], color=TXT, fontsize=9)
for bar, v in zip(bars, perf_df['Accuracy (%)']):
    ax1.text(bar.get_width()+0.5, bar.get_y()+bar.get_height()/2,
             f'{v}%', va='center', color=TXT, fontsize=9)

# ── AUC-ROC ────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
ax2.set_facecolor(BG_CARD)
auc_vals = pd.to_numeric(perf_df['AUC-ROC (%)'], errors='coerce').fillna(0)
auc_colors = ['#a29bfe' if v>=80 else '#fd79a8' if v>=65 else '#636e72' for v in auc_vals]
bars2 = ax2.barh(perf_df['Disease'], auc_vals, color=auc_colors, edgecolor='none', height=0.6)
ax2.set_xlim(0, 112)
ax2.set_xlabel('AUC-ROC (%)', color=TXT_DIM)
ax2.set_title('AUC-ROC Score', color=TXT, fontweight='bold', pad=10)
ax2.tick_params(colors=TXT_DIM); ax2.spines[:].set_visible(False)
ax2.set_yticklabels(perf_df['Disease'], color=TXT, fontsize=9)
for bar, v in zip(bars2, auc_vals):
    ax2.text(bar.get_width()+0.5, bar.get_y()+bar.get_height()/2,
             f'{v:.1f}%', va='center', color=TXT, fontsize=9)

# ── ROC Curves ────────────────────────────────────────────
ax3 = fig.add_subplot(gs[0, 2])
ax3.set_facecolor(BG_CARD)
roc_palette = plt.cm.tab10(np.linspace(0, 1, len(trained_models)))
for (key, data), color in zip(trained_models.items(), roc_palette):
    try:
        fpr, tpr, _ = roc_curve(data['y_test'], data['y_prob'])
        auc_v = roc_auc_score(data['y_test'], data['y_prob'])
        ax3.plot(fpr, tpr, color=color, lw=1.8,
                 label=f"{DISEASE_CONFIG[key]['display'][:12]} ({auc_v:.2f})")
    except: pass
ax3.plot([0,1],[0,1],'w--',lw=0.8,alpha=0.4)
ax3.set_xlabel('False Positive Rate', color=TXT_DIM)
ax3.set_ylabel('True Positive Rate', color=TXT_DIM)
ax3.set_title('ROC Curves — All Models', color=TXT, fontweight='bold', pad=10)
ax3.legend(fontsize=7, loc='lower right', framealpha=0.15,
           labelcolor=TXT, facecolor=BG_DARK, edgecolor='none')
ax3.tick_params(colors=TXT_DIM); ax3.spines[:].set_visible(False)

# ── Confusion Matrices (first 3 models) ───────────────────
disp_keys = list(trained_models.keys())[:3]
for plot_idx, key in enumerate(disp_keys):
    ax = fig.add_subplot(gs[1, plot_idx])
    ax.set_facecolor(BG_CARD)
    data = trained_models[key]
    y_pred_cm = (data['y_prob'] >= 0.5).astype(int)
    cm = confusion_matrix(data['y_test'], y_pred_cm)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                cbar=False, linewidths=0.5,
                xticklabels=['Negative','Positive'],
                yticklabels=['Negative','Positive'])
    ax.set_title(f'Confusion Matrix\n{DISEASE_CONFIG[key]["display"]}',
                 color=TXT, fontweight='bold', pad=8)
    ax.set_xlabel('Predicted', color=TXT_DIM)
    ax.set_ylabel('Actual', color=TXT_DIM)
    ax.tick_params(colors=TXT_DIM)

plt.suptitle('🏥 Gender-Aware Disease Prediction System — Model Performance Dashboard',
             color=TXT, fontsize=15, fontweight='bold', y=1.01)
plt.savefig('reports/model_performance_dashboard.png', bbox_inches='tight',
            facecolor=BG_DARK, dpi=140)
plt.show()
print('✅ Dashboard saved → reports/model_performance_dashboard.png')

## 🔬 CELL 8 — Feature Importance Analysis

In [ ]:
# ══════════════════════════════════════════════════════════
# CELL 8 ▸ Feature Importance — Top 3 Models
# ══════════════════════════════════════════════════════════

def extract_feature_importances(calibrated_model, n_features):
    """Extract averaged feature importances from a CalibratedClassifierCV."""
    importances = np.zeros(n_features)
    count = 0
    for cal_est in calibrated_model.calibrated_classifiers_:
        rf = cal_est.estimator.named_steps['classifier']
        importances += rf.feature_importances_
        count += 1
    return importances / count if count > 0 else importances


BG_DARK = '#0f1117'
BG_CARD = '#1a1d2e'

focus_keys = ['breast', 'diabetes', 'heart', 'pcos', 'thyroid', 'kidney']
focus_keys = [k for k in focus_keys if k in trained_models]

n_cols = 3
n_rows = (len(focus_keys) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 5*n_rows), facecolor=BG_DARK)
axes = np.array(axes).flatten()

for idx, key in enumerate(focus_keys):
    ax = axes[idx]
    ax.set_facecolor(BG_CARD)
    data     = trained_models[key]
    features = data['features']
    try:
        imps = extract_feature_importances(data['model'], len(features))
        top_n    = min(12, len(features))
        top_idx  = np.argsort(imps)[-top_n:]
        top_feat = [features[i] for i in top_idx]
        top_imp  = [imps[i] for i in top_idx]
        bar_colors = plt.cm.viridis(np.linspace(0.2, 0.9, top_n))
        ax.barh(range(top_n), top_imp, color=bar_colors, edgecolor='none', height=0.7)
        ax.set_yticks(range(top_n))
        ax.set_yticklabels([f[:25] for f in top_feat], color='white', fontsize=8)
        ax.set_xlabel('Importance Score', color='#adb5bd', fontsize=9)
        ax.set_title(f'{DISEASE_CONFIG[key]["display"]}\nTop {top_n} Features',
                     color='white', fontweight='bold', pad=8)
        ax.tick_params(colors='#adb5bd')
        ax.spines[:].set_visible(False)
    except Exception as e:
        ax.text(0.5, 0.5, f'N/A: {str(e)[:40]}', ha='center', va='center',
                color='#adb5bd', transform=ax.transAxes)
        ax.set_title(DISEASE_CONFIG[key]['display'], color='white', fontweight='bold')

# Hide unused axes
for i in range(len(focus_keys), len(axes)):
    axes[i].set_visible(False)

plt.suptitle('🔬 Feature Importance — Key Predictors per Disease',
             color='white', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('reports/feature_importance.png', bbox_inches='tight', facecolor=BG_DARK, dpi=140)
plt.show()
print('✅ Feature importance chart saved → reports/feature_importance.png')

## ⚙️ CELL 9 — Gender-Aware Risk Engine

In [ ]:
# ══════════════════════════════════════════════════════════
# CELL 9 ▸ Gender-Aware Risk Prediction Engine
# ══════════════════════════════════════════════════════════

class GenderAwareRiskEngine:
    """
    Full prediction pipeline:
      patient_data  →  per-disease probabilities
                    →  gender-weighted scores
                    →  overall health risk score
                    →  health report
    """

    # Common medical feature name aliases
    ALIASES = {
        'age':               ['age','Age','AGE','age of the patient'],
        'bmi':               ['bmi','BMI','body mass index (bmi)','body_mass_index'],
        'blood_pressure':    ['blood_pressure','bp','BloodPressure','trestbps',
                              'bp _systolic (mmhg)','blood pressure (mm/hg)'],
        'glucose':           ['glucose','Glucose','random blood glucose level (mg/dl)',
                              'rbs(mg/dl)'],
        'cholesterol':       ['cholesterol','chol','cholesterol level','Cholesterol'],
        'smoking':           ['smoking','Smoking','SMOKING','smoker',
                              'smoking status','smoking_yes'],
        'alcohol':           ['alcohol','Alcohol','ALCOHOL'],
        'physical_activity': ['physical_activity','PhysicalActivity',
                              'physical activity level','reg.exercise(y/n)'],
        'family_history':    ['family_history','FamilyHistory',
                              'family history of chronic kidney disease',
                              'family_history_with_overweight'],
        'hemoglobin':        ['hemoglobin','hb(g/dl)','hemoglobin level (gms)'],
        'insulin':           ['insulin','Insulin'],
        'pregnancies':       ['pregnancies','Pregnancies'],
        'skin_thickness':    ['skin_thickness','SkinThickness'],
        'tsh':               ['tsh','TSH (mIU/L)','tsh (miu/l)'],
    }

    def __init__(self, trained_models, disease_config, gender_weights):
        self.models  = trained_models
        self.config  = disease_config
        self.weights = gender_weights

    def _build_feature_vector(self, key, patient_data):
        """Map patient_data dict → model feature vector."""
        features = self.models[key]['features']
        flat     = {k.lower().replace(' ','_').replace('(','').replace(')','').strip(): v
                    for k,v in patient_data.items()}

        vec = []
        for feat in features:
            feat_key = feat.lower().replace(' ','_').replace('(','').replace(')','').strip()
            val = None

            # 1. Direct key match
            if feat_key in flat:
                val = flat[feat_key]
            else:
                # 2. Alias match
                for canon, alias_list in self.ALIASES.items():
                    norm_aliases = [a.lower().replace(' ','_').replace('(','').replace(')','').strip()
                                    for a in alias_list]
                    if feat_key in norm_aliases:
                        val = flat.get(canon, None)
                        break
                # 3. Partial match
                if val is None:
                    for pk in flat:
                        if pk in feat_key or feat_key in pk:
                            val = flat[pk]
                            break
            vec.append(float(val) if val is not None else 0.0)

        return np.array(vec).reshape(1, -1)

    def predict_all_diseases(self, patient_data):
        """Run all models and return raw {disease: probability} dict."""
        raw = {}
        for key in self.models:
            try:
                fv   = self._build_feature_vector(key, patient_data)
                prob = self.models[key]['model'].predict_proba(fv)[0][1]
                raw[key] = round(float(prob), 4)
            except Exception as e:
                raw[key] = 0.0
        return raw

    def apply_gender_weights(self, raw_risks, gender):
        """Multiply raw risk by gender-specific importance weights."""
        w_map = self.weights.get(gender.lower(), self.weights['male'])
        return {d: round(raw_risks[d] * w_map.get(d, 0.5), 4) for d in raw_risks}

    def compute_overall_score(self, weighted, gender):
        """Weighted average → normalized 0-100% overall health risk."""
        w_map  = self.weights.get(gender.lower(), self.weights['male'])
        total_w = sum(v for v in w_map.values() if v > 0)
        if total_w == 0: return 0.0
        wsum = sum(weighted.get(d, 0) for d in w_map if w_map[d] > 0)
        return round((wsum / total_w) * 100, 1)

    def generate_report(self, patient_data):
        """Full pipeline: patient_data → structured risk report dict."""
        gender   = patient_data.get('gender','male').lower()
        raw      = self.predict_all_diseases(patient_data)
        weighted = self.apply_gender_weights(raw, gender)
        overall  = self.compute_overall_score(weighted, gender)
        level, emoji, color = get_risk_level(overall)
        return {
            'patient':        patient_data,
            'gender':         gender,
            'raw_risks':      raw,
            'weighted_risks': weighted,
            'overall_score':  overall,
            'risk_level':     level,
            'emoji':          emoji,
            'color':          color
        }


engine = GenderAwareRiskEngine(trained_models, DISEASE_CONFIG, GENDER_WEIGHTS)
print('✅ Gender-Aware Risk Engine ready!')
print(f'   Loaded models: {list(engine.models.keys())}')

## 🖨️ CELL 10 — Visual Health Report Generator

In [ ]:
# ══════════════════════════════════════════════════════════
# CELL 10 ▸ Professional Visual Health Report Generator
# ══════════════════════════════════════════════════════════

def generate_health_report(report, save_path=None):
    gender   = report['gender']
    raw      = report['raw_risks']
    weighted = report['weighted_risks']
    overall  = report['overall_score']
    level    = report['risk_level']
    color    = report['color']
    patient  = report['patient']
    w_map    = GENDER_WEIGHTS[gender]

    # Filter to gender-relevant diseases only
    rel_keys  = [k for k in raw if w_map.get(k,0) > 0]
    rel_names = [DISEASE_CONFIG[k]['display'] for k in rel_keys]
    raw_pct   = [raw[k]*100  for k in rel_keys]
    w_pct     = [weighted[k]*100 for k in rel_keys]
    bar_colors= ['#e74c3c' if v>60 else '#f39c12' if v>35 else '#2ecc71' for v in raw_pct]

    BG  = '#0f1117'
    CARD= '#1a1d2e'
    TXT = 'white'

    fig = plt.figure(figsize=(20, 13), facecolor=BG)
    gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.50, wspace=0.35)

    # ── 1. Header ──────────────────────────────────────────
    ax_hdr = fig.add_subplot(gs[0, :])
    ax_hdr.set_facecolor('#12152a')
    ax_hdr.set_xlim(0,1); ax_hdr.set_ylim(0,1); ax_hdr.axis('off')
    g_icon = '♀' if gender=='female' else '♂'
    ax_hdr.text(0.5, 0.78, f'🏥  Personal Health Risk Report  {g_icon}',
                ha='center', color=TXT, fontsize=20, fontweight='bold')
    info = (f"Gender: {gender.capitalize()}   |   Age: {patient.get('age','N/A')}   |"
            f"   BMI: {patient.get('bmi','N/A')}   |   Glucose: {patient.get('glucose','N/A')}   |"
            f"   BP: {patient.get('blood_pressure','N/A')}")
    ax_hdr.text(0.5, 0.42, info, ha='center', color='#adb5bd', fontsize=11)
    # Color band for risk level
    band = mpatches.FancyBboxPatch((0.02,0.06), 0.96, 0.14,
                                    boxstyle='round,pad=0.01',
                                    facecolor=color, alpha=0.25,
                                    transform=ax_hdr.transAxes)
    ax_hdr.add_patch(band)
    ax_hdr.text(0.5, 0.13,
                f'Overall Health Risk Score:  {overall}%   [{level}] {report["emoji"]}',
                ha='center', color=color, fontsize=13, fontweight='bold')

    # ── 2. Raw Risk Bars ───────────────────────────────────
    ax1 = fig.add_subplot(gs[1, :2])
    ax1.set_facecolor(CARD)
    y_pos = range(len(rel_names))
    bars  = ax1.barh(y_pos, raw_pct, color=bar_colors, edgecolor='none', height=0.58)
    ax1.set_yticks(y_pos)
    ax1.set_yticklabels(rel_names, color=TXT, fontsize=10)
    ax1.set_xlabel('Risk Probability (%)', color='#adb5bd')
    ax1.set_title('Individual Disease Risk (Raw Model Output)', color=TXT, fontweight='bold', pad=9)
    ax1.set_xlim(0, 115); ax1.tick_params(colors='#adb5bd'); ax1.spines[:].set_visible(False)
    for bar, v in zip(bars, raw_pct):
        lbl = '🚨' if v>60 else '⚠️' if v>35 else '✅'
        ax1.text(bar.get_width()+0.5, bar.get_y()+bar.get_height()/2,
                 f'{lbl}  {v:.1f}%', va='center', color=TXT, fontsize=9)

    # ── 3. Gauge ───────────────────────────────────────────
    ax2 = fig.add_subplot(gs[1, 2])
    ax2.set_facecolor(CARD)
    ax2.set_xlim(-1.3,1.3); ax2.set_ylim(-0.25,1.3); ax2.axis('off')
    thetas_bg = np.linspace(np.pi, 0, 300)
    for i in range(len(thetas_bg)-1):
        frac = i/len(thetas_bg)
        ax2.plot([np.cos(thetas_bg[i]), np.cos(thetas_bg[i+1])],
                 [np.sin(thetas_bg[i]), np.sin(thetas_bg[i+1])],
                 color=plt.cm.RdYlGn_r(frac), lw=20, solid_capstyle='butt', alpha=0.75)
    needle_ang = np.pi - (overall/100)*np.pi
    ax2.annotate('', xy=(0.72*np.cos(needle_ang), 0.72*np.sin(needle_ang)),
                 xytext=(0,0),
                 arrowprops=dict(arrowstyle='->', color=color, lw=3.5))
    ax2.text(0, -0.08, f'{overall}%', ha='center', color=color, fontsize=24, fontweight='bold')
    ax2.text(0, -0.32, f'{level} RISK', ha='center', color=color, fontsize=14, fontweight='bold')
    ax2.set_title('Overall Health Risk', color=TXT, fontweight='bold', pad=10)

    # ── 4. Raw vs Weighted Comparison ─────────────────────
    ax3 = fig.add_subplot(gs[2, :2])
    ax3.set_facecolor(CARD)
    x = np.arange(len(rel_names))
    w = 0.38
    ax3.bar(x-w/2, raw_pct, w, label='Raw Risk %',             color='#74b9ff', alpha=0.8)
    ax3.bar(x+w/2, w_pct,   w, label='Gender-Weighted Risk %', color='#fd79a8', alpha=0.8)
    ax3.set_xticks(x)
    ax3.set_xticklabels([n[:14] for n in rel_names], rotation=30, ha='right', color=TXT, fontsize=9)
    ax3.set_ylabel('Risk %', color='#adb5bd')
    ax3.set_title(f'Raw vs Gender-Weighted Risk  ({gender.capitalize()} Prioritization)',
                  color=TXT, fontweight='bold', pad=9)
    ax3.legend(facecolor=BG, labelcolor=TXT, edgecolor='none', fontsize=9)
    ax3.tick_params(colors='#adb5bd'); ax3.spines[:].set_visible(False)

    # ── 5. Clinical Recommendations ───────────────────────
    ax4 = fig.add_subplot(gs[2, 2])
    ax4.set_facecolor(CARD); ax4.axis('off')
    ax4.set_title('📌 Clinical Recommendations', color=TXT, fontweight='bold', pad=9)
    sorted_risks = sorted(zip(rel_keys, raw_pct), key=lambda x: x[1], reverse=True)
    recs = []
    for k, v in sorted_risks[:4]:
        if v > 60:
            recs.append((f'🚨 {DISEASE_CONFIG[k]["display"]}', f'{v:.0f}% — Consult doctor urgently', '#e74c3c'))
        elif v > 35:
            recs.append((f'⚠️  {DISEASE_CONFIG[k]["display"]}', f'{v:.0f}% — Monitor & lifestyle change', '#f39c12'))
        else:
            recs.append((f'✅ {DISEASE_CONFIG[k]["display"]}', f'{v:.0f}% — Low risk, maintain habits', '#2ecc71'))
    for i, (title, desc, c) in enumerate(recs):
        ypos = 0.88 - i*0.22
        ax4.text(0.04, ypos,  title, transform=ax4.transAxes, color=c,  fontsize=10, fontweight='bold', va='top')
        ax4.text(0.04, ypos-0.09, desc,  transform=ax4.transAxes, color='#dfe6e9', fontsize=9,  va='top')

    if save_path:
        plt.savefig(save_path, bbox_inches='tight', facecolor=BG, dpi=150)
        print(f'💾 Report saved → {save_path}')
    plt.show()

print('✅ Report generator ready!')

## 🧪 CELL 11 — Predict: Female Patient

In [ ]:
# ══════════════════════════════════════════════════════════
# CELL 11 ▸ Female Patient — Prediction & Report
# ══════════════════════════════════════════════════════════
# 📝 Modify these values to test with different patients.

female_patient = {
    # ── Common features ─────────────────────────────────
    'gender':            'female',
    'age':               34,
    'bmi':               27.5,
    'blood_pressure':    80,          # diastolic (mm/Hg)
    'glucose':           110,         # mg/dl
    'cholesterol':       210,         # mg/dl
    'smoking':           0,           # 0=No, 1=Yes
    'alcohol':           0,
    'physical_activity': 1,           # 1=Active, 0=Sedentary
    'family_history':    1,           # 1=Yes, 0=No
    # ── Diabetes-specific ────────────────────────────────
    'pregnancies':       2,
    'insulin':           80,
    'skin_thickness':    20,
    'DiabetesPedigreeFunction': 0.35,
    # ── PCOS-specific ────────────────────────────────────
    'weight gain(y/n)':  1,
    'hair growth(y/n)':  0,
    'skin darkening (y/n)': 0,
    'pimples(y/n)':      0,
    'cycle(r/i)':        1,
    'tsh':               2.5,
    # ── Thyroid-specific ─────────────────────────────────
    'smoking_yes':       0,
    'risk_low':          1,
    # ── Breast-specific ──────────────────────────────────
    'radius_mean':       14.5,
    'texture_mean':      18.0,
    'perimeter_mean':    93.0,
    'area_mean':         650.0,
    'smoothness_mean':   0.096,
    'concavity_mean':    0.07,
    # ── Heart-specific ───────────────────────────────────
    'cp':       0,
    'trestbps': 120,
    'chol':     210,
    'fbs':      0,
    'thalach':  150,
    'exang':    0,
    'oldpeak':  1.0,
    'ca':       0,
    'thal':     2,
    # ── Liver-specific ───────────────────────────────────
    'total_bilirubin':   0.9,
    'albumin':           3.8,
}

# ── Run prediction ────────────────────────────────────────
female_report = engine.generate_report(female_patient)

# ── Console summary ──────────────────────────────────────
print('═'*58)
print('       🏥 HEALTH RISK REPORT  ♀  FEMALE PATIENT')
print('═'*58)
print(f'  Age: {female_patient["age"]}  |  BMI: {female_patient["bmi"]}  |  Glucose: {female_patient["glucose"]}')
print('─'*58)

w_map = GENDER_WEIGHTS['female']
sorted_risks = sorted(female_report['raw_risks'].items(),
                       key=lambda x: w_map.get(x[0],0)*x[1], reverse=True)
for key, prob in sorted_risks:
    if w_map.get(key,0) == 0: continue
    pct  = prob*100
    icon = '🚨' if pct>60 else '⚠️ ' if pct>35 else '✅'
    bar  = '█'*int(pct//5) + '░'*(20-int(pct//5))
    wt   = w_map.get(key,0.5)
    print(f'  {icon} {DISEASE_CONFIG[key]["display"]:20s}  {bar}  {pct:5.1f}%  [w={wt}]')

print('─'*58)
print(f'  OVERALL RISK SCORE : {female_report["overall_score"]}%')
print(f'  RISK LEVEL         : {female_report["emoji"]}  {female_report["risk_level"]}')
print('═'*58)

generate_health_report(female_report, save_path='reports/report_female.png')

## 🧪 CELL 12 — Predict: Male Patient

In [ ]:
# ══════════════════════════════════════════════════════════
# CELL 12 ▸ Male Patient — Prediction & Report
# ══════════════════════════════════════════════════════════

male_patient = {
    'gender':            'male',
    'age':               52,
    'bmi':               30.1,
    'blood_pressure':    90,
    'glucose':           145,
    'cholesterol':       265,
    'smoking':           1,
    'alcohol':           1,
    'physical_activity': 0,
    'family_history':    1,
    # Heart-specific
    'cp':       2,
    'trestbps': 140,
    'chol':     265,
    'fbs':      1,
    'thalach':  140,
    'exang':    1,
    'oldpeak':  2.5,
    'slope':    1,
    'ca':       2,
    'thal':     2,
    'sex':      1,
    # Diabetes-specific
    'Pregnancies':       0,
    'Insulin':           130,
    'SkinThickness':     30,
    'DiabetesPedigreeFunction': 0.62,
    # Liver-specific
    'total_bilirubin':   1.8,
    'direct_bilirubin':  0.6,
    'alkaline_phosphotase': 120,
    'alamine_aminotransferase': 55,
    'aspartate_aminotransferase': 48,
    'albumin':           3.2,
    # Prostate-specific
    'lcavol':  2.1,
    'lweight': 3.8,
    'lbph':    0.3,
    'svi':     1,
    'lcp':     1.5,
    'gleason': 7,
    'pgg45':   40,
    # Kidney-specific
    'albumin in urine':  2,
    'sugar in urine':    0,
    'hemoglobin':        11.0,
    # Lung-specific
    'lung capacity':     3.2,
    'hospital visits':   5,
    'disease type':      1,
    'treatment type':    1,
}

male_report = engine.generate_report(male_patient)

print('═'*58)
print('       🏥 HEALTH RISK REPORT  ♂  MALE PATIENT')
print('═'*58)
print(f'  Age: {male_patient["age"]}  |  BMI: {male_patient["bmi"]}  |  Glucose: {male_patient["glucose"]}')
print('─'*58)

w_map = GENDER_WEIGHTS['male']
sorted_risks = sorted(male_report['raw_risks'].items(),
                       key=lambda x: w_map.get(x[0],0)*x[1], reverse=True)
for key, prob in sorted_risks:
    if w_map.get(key,0) == 0: continue
    pct  = prob*100
    icon = '🚨' if pct>60 else '⚠️ ' if pct>35 else '✅'
    bar  = '█'*int(pct//5) + '░'*(20-int(pct//5))
    wt   = w_map.get(key,0.5)
    print(f'  {icon} {DISEASE_CONFIG[key]["display"]:20s}  {bar}  {pct:5.1f}%  [w={wt}]')

print('─'*58)
print(f'  OVERALL RISK SCORE : {male_report["overall_score"]}%')
print(f'  RISK LEVEL         : {male_report["emoji"]}  {male_report["risk_level"]}')
print('═'*58)

generate_health_report(male_report, save_path='reports/report_male.png')

## 🎛️ CELL 13 — Interactive Widget (Colab Form)

In [ ]:
# ══════════════════════════════════════════════════════════
# CELL 13 ▸ Symptom-Based Risk Prediction Widget
# ══════════════════════════════════════════════════════════

try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
    import datetime

    S  = {'description_width': '230px'}
    LW = widgets.Layout(width='460px')
    LH = widgets.Layout(width='460px', height='44px')

    def dd(desc, opts):
        return widgets.Dropdown(description=desc, options=opts, style=S, layout=LW)
    def sl_i(desc, val, mn, mx, step=1):
        return widgets.IntSlider(value=val, min=mn, max=mx, step=step,
                                  description=desc, style=S, layout=LW)
    def sl_f(desc, val, mn, mx, step=0.1):
        return widgets.FloatSlider(value=val, min=mn, max=mx, step=step,
                                    description=desc, style=S, layout=LW)
    def hdr(text, color='#74b9ff'):
        return widgets.HTML(
            f'<div style="background:#12152a;border-left:4px solid {color};'
            f'padding:8px 12px;margin:8px 0 4px;border-radius:4px">'
            f'<b style="color:{color};font-family:sans-serif;font-size:13px">{text}</b></div>')
    def note(text):
        return widgets.HTML(
            f'<p style="color:#adb5bd;font-size:11px;font-family:sans-serif;'
            f'margin:0 0 4px 4px;font-style:italic">ℹ️  {text}</p>')

    YES_NO   = [('No', 0), ('Yes', 1)]
    SEVERITY = [('Never / None', 0), ('Mild / Occasional', 1),
                ('Moderate / Often', 2), ('Severe / Always', 3)]

    # ── TAB 0: About You ──────────────────────────────────
    w_name    = widgets.Text(value='', placeholder='e.g. Priya Sharma',
                             description='Your Name:', style=S, layout=LW)
    w_gender  = dd('Gender:', [('Female', 'female'), ('Male', 'male')])
    w_age     = sl_i('Age (years):', 35, 18, 90)
    w_height  = sl_i('Height (cm):', 160, 100, 220)
    w_weight  = sl_f('Weight (kg):', 65.0, 30, 180, step=0.5)
    w_bmi_lbl = widgets.HTML('')
    w_waist   = sl_i('Waist Size (inches):', 32, 24, 60)
    w_family  = dd('Family History of Disease:',
                   [('None', 0), ('One parent/sibling', 1), ('Both parents', 2)])
    w_stress  = dd('Daily Stress Level:',
                   [('Low', 0), ('Moderate', 1), ('High', 2), ('Very High', 3)])
    w_sleep   = dd('Sleep Quality:',
                   [('Good — 7-9 hrs restful', 0),
                    ('Fair — 5-7 hrs', 1),
                    ('Poor — under 5 hrs', 2)])

    def _bmi_update(change):
        h   = w_height.value / 100
        bmi = round(w_weight.value / (h * h), 1)
        cat = ('Underweight' if bmi < 18.5 else 'Normal' if bmi < 25
               else 'Overweight' if bmi < 30 else 'Obese')
        col = '#2ecc71' if bmi < 25 else '#f39c12' if bmi < 30 else '#e74c3c'
        w_bmi_lbl.value = (f'<p style="color:{col};font-family:sans-serif;'
                           f'font-size:13px;margin:0 0 6px 4px">'
                           f'<b>BMI: {bmi} — {cat}</b></p>')
    w_height.observe(_bmi_update, names='value')
    w_weight.observe(_bmi_update, names='value')
    _bmi_update(None)

    tab0 = widgets.VBox([
        hdr('👤 About You'),
        w_name, w_gender, w_age, w_height, w_weight, w_bmi_lbl, w_waist,
        hdr('🧬 Background'),
        w_family, w_stress, w_sleep,
    ])

    # ── TAB 1: Lifestyle ──────────────────────────────────
    w_smoking   = dd('Do you smoke?', YES_NO)
    w_smoke_yrs = sl_i('Smoking years (if yes):', 0, 0, 50)
    w_alcohol   = dd('Alcohol intake:',
                     [('No', 0), ('Occasionally', 1), ('Regularly', 2), ('Daily/Heavy', 3)])
    w_activity  = dd('Physical Activity:',
                     [('Very Active — daily', 3), ('Active — 3-4x/week', 2),
                      ('Light — 1-2x/week', 1), ('Sedentary — rarely', 0)])
    w_diet      = dd('Typical Diet:',
                     [('Balanced', 0), ('Mostly home cooked', 1),
                      ('Frequent junk food', 2), ('Very unhealthy', 3)])
    w_water     = dd('Water intake/day:',
                     [('8+ glasses', 0), ('5-8 glasses', 1), ('Less than 5', 2)])
    w_outdoor   = dd('Sun/outdoor exposure:',
                     [('Daily 30+ min', 0), ('Sometimes', 1), ('Rarely', 2)])

    tab1 = widgets.VBox([
        hdr('🚬 Smoking & Alcohol'),
        w_smoking, w_smoke_yrs, w_alcohol,
        hdr('🏃 Activity & Diet'),
        w_activity, w_diet, w_water, w_outdoor,
    ])

    # ── TAB 2: General Symptoms ───────────────────────────
    w_fatigue    = dd('Fatigue / Always tired:', SEVERITY)
    w_weakness   = dd('Body weakness:', SEVERITY)
    w_fever      = dd('Frequent fever / infections:', SEVERITY)
    w_weightloss = dd('Unexplained weight loss:', YES_NO)
    w_weightgain = dd('Unexplained weight gain:', YES_NO)
    w_appetite   = dd('Loss of appetite:', SEVERITY)
    w_nausea     = dd('Nausea / Vomiting:', SEVERITY)
    w_swelling   = dd('Swelling in legs / feet / face:', SEVERITY)
    w_nightsweat = dd('Night sweats:', YES_NO)
    w_skinchange = dd('Skin color changes (yellow/dark):', SEVERITY)
    w_itching    = dd('Itching / skin rash:', SEVERITY)
    w_hairloss   = dd('Hair loss / thinning:', YES_NO)
    w_anemia_sym = dd('Feeling unusually pale / anemic:', YES_NO)

    tab2 = widgets.VBox([
        hdr('😷 General Body Symptoms', '#e74c3c'),
        w_fatigue, w_weakness, w_fever,
        w_weightloss, w_weightgain, w_appetite,
        w_nausea, w_swelling, w_nightsweat,
        w_skinchange, w_itching, w_hairloss, w_anemia_sym,
    ])

    # ── TAB 3: Heart & Lungs ──────────────────────────────
    w_chestpain   = dd('Chest pain or tightness:', SEVERITY)
    w_exercise_cp = dd('Chest pain during exercise:', SEVERITY)
    w_breathless  = dd('Shortness of breath:', SEVERITY)
    w_palpitation = dd('Heart palpitations / racing heart:', SEVERITY)
    w_dizziness   = dd('Dizziness / lightheadedness:', SEVERITY)
    w_bp_high     = dd('Known high blood pressure?',
                       [('No', 0), ('Borderline', 1),
                        ('Yes — controlled', 2), ('Yes — uncontrolled', 3)])
    w_leg_cramps  = dd('Leg cramps / pain while walking:', SEVERITY)
    w_cough       = dd('Persistent cough:', SEVERITY)
    w_cough_blood = dd('Coughing up blood:', YES_NO)
    w_wheeze      = dd('Wheezing / noisy breathing:', SEVERITY)
    w_rapid_heart = dd('Rapid heartbeat / tremors:', YES_NO)

    tab3 = widgets.VBox([
        hdr('❤️ Heart & Chest', '#e74c3c'),
        note('Chest pain, breathlessness and high BP are key heart disease indicators.'),
        w_chestpain, w_exercise_cp, w_breathless,
        w_palpitation, w_dizziness, w_bp_high, w_leg_cramps, w_rapid_heart,
        hdr('🫁 Respiratory', '#74b9ff'),
        w_cough, w_cough_blood, w_wheeze,
    ])

    # ── TAB 4: Diabetes & Kidney ──────────────────────────
    w_thirst      = dd('Excessive thirst:', SEVERITY)
    w_urination   = dd('Frequent urination:', SEVERITY)
    w_blurred_vis = dd('Blurred vision:', SEVERITY)
    w_slow_heal   = dd('Slow healing wounds / cuts:', SEVERITY)
    w_tingling    = dd('Tingling / numbness in hands/feet:', SEVERITY)
    w_dark_skin   = dd('Dark patches on skin (neck/armpits):', YES_NO)
    w_diabetes_hx = dd('Diagnosed with diabetes before?', YES_NO)
    w_urine_foam  = dd('Foamy / frothy urine:', YES_NO)
    w_urine_blood = dd('Blood in urine:', YES_NO)
    w_back_pain   = dd('Back / flank pain (kidney area):', SEVERITY)
    w_urine_less  = dd('Reduced urine output:', YES_NO)

    tab4 = widgets.VBox([
        hdr('🍬 Diabetes Symptoms', '#f39c12'),
        note('Thirst, frequent urination and slow healing are classic diabetes warning signs.'),
        w_thirst, w_urination, w_blurred_vis,
        w_slow_heal, w_tingling, w_dark_skin, w_diabetes_hx,
        hdr('🫘 Kidney Symptoms', '#a29bfe'),
        w_urine_foam, w_urine_blood, w_back_pain, w_urine_less,
    ])

    # ── TAB 5: Liver & Thyroid ────────────────────────────
    w_yellow_skin  = dd('Yellow skin / eyes (jaundice):', SEVERITY)
    w_abdpain      = dd('Upper right abdomen pain:', SEVERITY)
    w_dark_urine   = dd('Dark colored urine:', YES_NO)
    w_pale_stool   = dd('Pale / clay colored stool:', YES_NO)
    w_liver_alc    = dd('Heavy alcohol use (past/present):', YES_NO)
    w_cold_intol   = dd('Always feeling cold:', YES_NO)
    w_heat_intol   = dd('Always feeling hot / sweating:', YES_NO)
    w_thyroid_swell= dd('Swelling in front of neck:', YES_NO)
    w_voice_change = dd('Hoarse / changed voice:', YES_NO)
    w_constipation = dd('Chronic constipation:', SEVERITY)
    w_memory       = dd('Memory / concentration issues:', SEVERITY)

    tab5 = widgets.VBox([
        hdr('🫀 Liver Symptoms', '#f39c12'),
        note('Jaundice and dark urine are strong liver disease indicators.'),
        w_yellow_skin, w_abdpain, w_dark_urine, w_pale_stool, w_liver_alc,
        hdr('🦋 Thyroid Symptoms', '#2ecc71'),
        note('Neck swelling, temperature intolerance and hair loss suggest thyroid issues.'),
        w_cold_intol, w_heat_intol, w_thyroid_swell,
        w_voice_change, w_constipation, w_memory,
    ])

    # ── TAB 6: Female / Male ──────────────────────────────
    w_irreg_cycle  = dd('Irregular / missed periods:', SEVERITY)
    w_heavy_period = dd('Heavy or painful periods:', SEVERITY)
    w_facial_hair  = dd('Excess facial / body hair:', YES_NO)
    w_pcos_acne    = dd('Acne / oily skin:', SEVERITY)
    w_pelvic_pain  = dd('Pelvic pain:', SEVERITY)
    w_infertility  = dd('Difficulty conceiving:', YES_NO)
    w_preg         = dd('Currently pregnant:', YES_NO)
    w_urine_weak   = dd('Weak urine stream:', SEVERITY)
    w_urine_freq_m = dd('Frequent nighttime urination:', SEVERITY)
    w_urine_pain   = dd('Pain/burning during urination:', SEVERITY)
    w_prostate_pain= dd('Pain in lower back / hips / pelvis:', SEVERITY)
    w_blood_semen  = dd('Blood in semen / urine:', YES_NO)
    w_erectile     = dd('Erectile difficulties:', YES_NO)

    tab6 = widgets.VBox([
        hdr('🌸 Female / PCOS Symptoms', '#fd79a8'),
        note('Irregular periods, excess hair and acne together strongly indicate PCOS.'),
        w_irreg_cycle, w_heavy_period, w_facial_hair,
        w_pcos_acne, w_pelvic_pain, w_infertility, w_preg,
        hdr('🔵 Male / Prostate Symptoms', '#74b9ff'),
        note('Weak stream and nighttime urination are key prostate risk indicators.'),
        w_urine_weak, w_urine_freq_m, w_urine_pain,
        w_prostate_pain, w_blood_semen, w_erectile,
    ])

    # ── Tabs ──────────────────────────────────────────────
    tab = widgets.Tab(children=[tab0, tab1, tab2, tab3, tab4, tab5, tab6])
    for i, t in enumerate(['👤 About You', '🏃 Lifestyle', '😷 General',
                            '❤️ Heart/Lungs', '🍬 Diabetes/Kidney',
                            '🫀 Liver/Thyroid', '⚥ Female/Male']):
        tab.set_title(i, t)

    btn_generate = widgets.Button(description='🔍  Predict My Health Risk',
                                   button_style='success', layout=LH)
    btn_download = widgets.Button(description='⬇️  Download Report',
                                   button_style='info', layout=LH, disabled=True)
    output_area  = widgets.Output()
    last_report  = {'path': None, 'downloaded': False}

    # ── Symptom → Clinical Feature Converter ─────────────
    def symptoms_to_features():
        gender   = w_gender.value
        is_male  = 1 if gender == 'male' else 0
        age      = w_age.value
        h        = w_height.value / 100
        bmi      = round(w_weight.value / (h * h), 1)
        smoking  = w_smoking.value
        alcohol  = min(1, w_alcohol.value)
        activity = 1 if w_activity.value >= 1 else 0
        act_num  = w_activity.value
        family   = min(1, w_family.value)
        stress   = w_stress.value
        diet_bad = w_diet.value

        # Read all symptom values once here (no global needed)
        fatigue_s    = w_fatigue.value
        breathless_s = w_breathless.value
        chest_s      = w_chestpain.value
        thirst_s     = w_thirst.value
        urination_s  = w_urination.value
        swelling_s   = w_swelling.value
        yellow_s     = w_yellow_skin.value
        cough_s      = w_cough.value
        back_pain_s  = w_back_pain.value
        anemia_s     = w_anemia_sym.value

        # ── Estimate clinical values from symptoms ────────
        bp_sys  = int(120 + w_bp_high.value*15 + smoking*8 + (bmi-25)*0.5
                      + stress*5 + (1-activity)*5 + chest_s*5)
        bp_dia  = int(bp_sys * 0.65)

        glucose = int(90 + thirst_s*20 + urination_s*15 + w_dark_skin.value*20
                      + w_diabetes_hx.value*30 + diet_bad*10
                      + (bmi-25)*1.2 + w_slow_heal.value*10)

        chol    = int(170 + diet_bad*25 + (bmi-25)*2 + smoking*15
                      + (1-activity)*15 + age*0.5)

        hb      = round(max(7.0, 15.0 - fatigue_s*0.8 - w_weightloss.value*1.0
                            - anemia_s*1.5 - alcohol*0.8), 1)

        insulin = int(max(10, glucose * 0.7 - 30 + thirst_s*10))

        creat   = round(max(0.5, 0.9 + w_urine_foam.value*1.5
                            + w_urine_blood.value*1.0 + back_pain_s*0.4
                            + swelling_s*0.3 + w_bp_high.value*0.2), 2)

        egfr    = int(max(5, 110 - creat*15 - (age-30)*0.3))
        urea    = int(15 + creat*10 + w_urine_less.value*20 + swelling_s*5)

        alc_heavy = w_alcohol.value >= 2
        alt     = int(20 + yellow_s*40 + w_abdpain.value*20
                      + alc_heavy*80 + w_liver_alc.value*60)
        ast     = int(18 + yellow_s*35 + w_abdpain.value*18
                      + alc_heavy*70 + w_liver_alc.value*50)
        alk_ph  = int(100 + yellow_s*100 + alc_heavy*150 + w_abdpain.value*50)
        tbili   = round(0.5 + yellow_s*2.5 + w_dark_urine.value*1.5, 1)
        dbili   = round(tbili * 0.4, 1)
        albumin = round(max(2.0, 4.2 - yellow_s*0.4 - alc_heavy*0.5 - fatigue_s*0.2), 1)
        t_prot  = round(albumin + 3.5 - w_appetite.value*0.3, 1)
        agr     = round(albumin / max(0.1, t_prot - albumin), 2)

        lung_cap = round(max(1.2, 5.0 - smoking*1.2 - w_smoke_yrs.value*0.02
                             - breathless_s*0.5 - (age-35)*0.015), 1)

        h_rate  = int(70 + w_palpitation.value*10 + w_rapid_heart.value*15
                      + stress*5 + smoking*5)

        cp_val  = (3 if chest_s >= 3 else 2 if chest_s == 2
                   else 1 if chest_s == 1 else 0)
        oldpeak = round(min(6.2, chest_s*0.8 + w_exercise_cp.value*0.6), 1)
        ca_val  = min(3, int(smoking + (1 if chol>240 else 0)
                             + (1 if w_bp_high.value>=2 else 0)
                             + (1 if w_exercise_cp.value>=2 else 0)))

        dpf     = round(0.2 + family*0.3 + w_diabetes_hx.value*0.4 + diet_bad*0.1, 2)
        skin_th = int(bmi*0.7 + (1-activity)*5 + thirst_s*3)

        tsh     = round(2.5 + w_cold_intol.value*3.0 + w_heat_intol.value*(-1.5)
                        + w_hairloss.value*1.5 + w_constipation.value*1.0
                        + smoking*0.5, 1)

        waist_in = w_waist.value
        hip_in   = int(waist_in / max(0.7, 0.76 + bmi*0.005))
        whr      = round(waist_in / max(1, hip_in), 2)

        lcavol  = round(max(-1.4, 0.5 + (age-50)*0.05 + smoking*0.4
                            + family*0.5 + w_urine_weak.value*0.3
                            + w_prostate_pain.value*0.4), 2)
        pgg45   = min(100, int(5 + w_urine_weak.value*15
                               + w_blood_semen.value*25 + family*20 + smoking*10))
        gleason = 6 + (1 if w_blood_semen.value else 0) + (1 if (family and smoking) else 0)

        risk_low   = 0 if (smoking or family or bmi>30 or w_thyroid_swell.value) else 1
        risk_inter = 1 if (smoking or family or w_thyroid_swell.value) else 0

        cycle_val = (w_irreg_cycle.value + 2) if gender == 'female' else 2
        lh_val    = round(6.0 + w_irreg_cycle.value*2 + w_facial_hair.value*3 + (bmi-25)*0.3, 1)
        fsh_val   = round(8.0 + w_heavy_period.value*2 + (1-activity)*2, 1)
        amh_val   = round(4.0 + w_irreg_cycle.value*3 + w_facial_hair.value*2, 1)
        foll_n    = int(6 + w_irreg_cycle.value*3 + w_facial_hair.value*2)

        return {
            'gender': gender, 'sex': is_male, 'Gender': is_male, 'Gender_M': is_male,
            'age': age, 'Age': age, 'age of the patient': age, ' Age (yrs)': age,
            'bmi': bmi, 'BMI': bmi, 'Body Mass Index (BMI)': bmi,
            'Weight (Kg)': w_weight.value, 'Height(Cm) ': w_height.value,
            'blood_pressure': bp_dia, 'trestbps': bp_sys, 'BloodPressure': bp_dia,
            'Blood pressure (mm/Hg)': bp_sys,
            'BP _Systolic (mmHg)': bp_sys, 'BP _Diastolic (mmHg)': bp_dia,
            'glucose': glucose, 'Glucose': glucose,
            'Random blood glucose level (mg/dl)': glucose, 'RBS(mg/dl)': glucose,
            'cholesterol': chol, 'chol': chol, 'Cholesterol level': chol,
            'Hemoglobin level (gms)': hb, 'Hb(g/dl)': hb, 'hemoglobin': hb,
            'Packed cell volume (%)': round(hb*3, 1),
            'White blood cell count (cells/cumm)': int(7000 + w_fever.value*2000),
            'Red blood cell count (millions/cumm)': round(max(2.5, hb/3.3), 1),
            'Insulin': insulin, 'SkinThickness': skin_th, 'DiabetesPedigreeFunction': dpf,
            'Serum creatinine (mg/dl)': creat,
            'Estimated Glomerular Filtration Rate (eGFR)': egfr,
            'Blood urea (mg/dl)': urea,
            'Sodium level (mEq/L)': 138, 'Potassium level (mEq/L)': 4.5,
            'Albumin in urine': min(5, w_urine_foam.value + w_bp_high.value),
            'Sugar in urine': min(5, int(glucose/80)),
            'Specific gravity of urine': 1.015 + (0.005 if glucose > 140 else 0),
            'Red blood cells in urine': w_urine_blood.value,
            'Pus cells in urine': 0, 'Pus cell clumps in urine': 0, 'Bacteria in urine': 0,
            'Hypertension (yes/no)': 1 if w_bp_high.value >= 2 else 0,
            'Diabetes mellitus (yes/no)': 1 if (glucose > 126 or w_diabetes_hx.value) else 0,
            'Coronary artery disease (yes/no)': 1 if (chol > 240 and bp_sys > 140) else 0,
            'Appetite (good/poor)': 0 if w_appetite.value >= 2 else 1,
            'Pedal edema (yes/no)': 1 if swelling_s >= 2 else 0,
            'Anemia (yes/no)': 1 if hb < 11 else 0,
            'Urine output (ml/day)': int(1800 - w_urine_less.value*500 - swelling_s*100),
            'Urine protein-to-creatinine ratio': round(creat*0.3, 2),
            'Cystatin C level': round(1.0 + (bp_sys-120)*0.01, 2),
            'C-reactive protein (CRP) level': round(1.5 + smoking*3 + w_fever.value*2, 1),
            'Interleukin-6 (IL-6) level': round(2.0 + smoking*3.5 + w_fever.value*2, 1),
            'Serum albumin level': albumin,
            'Parathyroid hormone (PTH) level': 40,
            'Serum calcium level': 9.0, 'Serum phosphate level': 4.2,
            'Family history of chronic kidney disease': family,
            'Smoking status': smoking, 'Physical activity level': act_num,
            'Duration of diabetes mellitus (years)': 5 if w_diabetes_hx.value else 0,
            'Duration of hypertension (years)': 5 if w_bp_high.value >= 2 else 0,
            'Urinary sediment microscopy results': w_urine_blood.value,
            'Pregnancies': 0 if gender == 'male' else w_preg.value,
            'cp': cp_val, 'fbs': 1 if glucose > 120 else 0,
            'restecg': 1 if w_bp_high.value >= 2 else 0,
            'thalach': h_rate, 'exang': 1 if w_exercise_cp.value >= 2 else 0,
            'oldpeak': oldpeak, 'slope': 1 if chest_s >= 2 else 0,
            'ca': ca_val, 'thal': 2,
            'Total_Bilirubin': tbili, 'Direct_Bilirubin': dbili,
            'Alkaline_Phosphotase': alk_ph,
            'Alamine_Aminotransferase': alt, 'Aspartate_Aminotransferase': ast,
            'Total_Protiens': t_prot, 'Albumin': albumin, 'Albumin_and_Globulin_Ratio': agr,
            'Lung Capacity': lung_cap,
            'Disease Type': min(4, int(cough_s + w_cough_blood.value*2)),
            'Treatment Type': 1 if cough_s >= 2 else 0,
            'Hospital Visits': int(2 + cough_s*2 + w_cough_blood.value*3),
            'lcavol': lcavol, 'lweight': round(3.5 + (bmi-25)*0.02, 2),
            'lbph': round(0.1 + (age-40)*0.015, 2),
            'svi': 1 if (w_blood_semen.value and family) else 0,
            'lcp': round(lcavol - 0.5, 2), 'gleason': gleason, 'pgg45': pgg45,
            'radius_mean': 14.13, 'texture_mean': 19.29,
            'perimeter_mean': 91.97, 'area_mean': 654.89,
            'smoothness_mean': 0.10, 'compactness_mean': 0.10,
            'concavity_mean': 0.09, 'concave points_mean': 0.05,
            'symmetry_mean': 0.18, 'fractal_dimension_mean': 0.06,
            'radius_se': 0.41, 'texture_se': 1.22,
            'perimeter_se': 2.87, 'area_se': 40.34,
            'smoothness_se': 0.01, 'compactness_se': 0.03,
            'concavity_se': 0.03, 'concave points_se': 0.01,
            'symmetry_se': 0.02, 'fractal_dimension_se': 0.003,
            'radius_worst': 16.27, 'texture_worst': 25.68,
            'perimeter_worst': 107.26, 'area_worst': 880.58,
            'smoothness_worst': 0.13, 'compactness_worst': 0.25,
            'concavity_worst': 0.27, 'concave points_worst': 0.11,
            'symmetry_worst': 0.29, 'fractal_dimension_worst': 0.08,
            'Blood Group': 11,
            'Pulse rate(bpm) ': h_rate, 'RR (breaths/min)': int(16 + breathless_s*3),
            'Cycle(R/I)': cycle_val, 'Cycle length(days)': int(5 + w_irreg_cycle.value),
            'Marraige Status (Yrs)': 3, 'Pregnant(Y/N)': w_preg.value,
            'No. of abortions': 0,
            '  I   beta-HCG(mIU/mL)': 5.0, 'II    beta-HCG(mIU/mL)': 5.0,
            'FSH(mIU/mL)': fsh_val, 'LH(mIU/mL)': lh_val,
            'FSH/LH': round(fsh_val / max(0.1, lh_val), 2),
            'Hip(inch)': hip_in, 'Waist(inch)': waist_in, 'Waist:Hip Ratio': whr,
            'TSH (mIU/L)': tsh, 'AMH(ng/mL)': amh_val, 'PRL(ng/mL)': 20.0,
            'Vit D3 (ng/mL)': max(5, 35 - (1-activity)*10 - w_outdoor.value*8),
            'PRG(ng/mL)': 0.5,
            'Weight gain(Y/N)': w_weightgain.value,
            'hair growth(Y/N)': w_facial_hair.value,
            'Skin darkening (Y/N)': w_dark_skin.value,
            'Hair loss(Y/N)': w_hairloss.value,
            'Pimples(Y/N)': 1 if w_pcos_acne.value >= 2 else 0,
            'Fast food (Y/N)': 1 if diet_bad >= 2 else 0,
            'Reg.Exercise(Y/N)': activity,
            'Follicle No. (L)': foll_n, 'Follicle No. (R)': foll_n,
            'Avg. F size (L) (mm)': 14.0, 'Avg. F size (R) (mm)': 14.0,
            'Endometrium (mm)': 8.0,
            'Smoking_Yes': smoking, 'Hx Smoking_Yes': smoking,
            'Hx Radiothreapy_Yes': 0,
            'Thyroid Function_Clinical Hypothyroidism': 1 if (tsh > 10 and w_cold_intol.value) else 0,
            'Thyroid Function_Euthyroid': 1 if tsh <= 5 else 0,
            'Thyroid Function_Subclinical Hyperthyroidism': 1 if (tsh < 0.5 and w_heat_intol.value) else 0,
            'Thyroid Function_Subclinical Hypothyroidism': 1 if (5 < tsh <= 10) else 0,
            'Physical Examination_Multinodular goiter': 0,
            'Physical Examination_Normal': 1 if not w_thyroid_swell.value else 0,
            'Physical Examination_Single nodular goiter-left': 0,
            'Physical Examination_Single nodular goiter-right': w_thyroid_swell.value,
            'Adenopathy_Extensive': 0, 'Adenopathy_Left': 0,
            'Adenopathy_No': 1 if not w_thyroid_swell.value else 0,
            'Adenopathy_Posterior': 0, 'Adenopathy_Right': 0,
            'Pathology_Hurthel cell': 0, 'Pathology_Micropapillary': 0,
            'Pathology_Papillary': 1, 'Focality_Uni-Focal': 1,
            'Risk_Intermediate': risk_inter, 'Risk_Low': risk_low,
            'T_T1b': 0, 'T_T2': 1, 'T_T3a': 0, 'T_T3b': 0,
            'T_T4a': 0, 'T_T4b': 0, 'N_N1a': 0, 'N_N1b': 0, 'M_M1': 0,
            'Stage_II': 0, 'Stage_III': 0, 'Stage_IVA': 0, 'Stage_IVB': 0,
            'Response_Excellent': 1, 'Response_Indeterminate': 0,
            'Response_Structural Incomplete': 0,
            'smoking': smoking, 'Smoking': smoking,
            'alcohol': alcohol, 'physical_activity': activity, 'family_history': family,
        }

    # ── Generate button handler ───────────────────────────
    def on_predict(b):
        with output_area:
            clear_output(wait=True)
            name     = w_name.value.strip() or 'Anonymous Patient'
            now      = datetime.datetime.now()
            dt_label = now.strftime('%Y/%m/%d-%I:%M %p')
            dt_stamp = now.strftime('%Y%m%d_%H%M%S')

            print(f'🔄 Running prediction for {name}...')
            patient = symptoms_to_features()
            rep     = engine.generate_report(patient)

            safe_name = name.replace(' ', '_')
            img_path  = f'reports/health_report_{safe_name}_{dt_stamp}.png'
            generate_health_report(rep, save_path=img_path)

            last_report['path']       = img_path
            last_report['downloaded'] = False
            btn_download.disabled     = False
            btn_download.description  = '⬇️  Download Report'

            print('═' * 50)
            print(f'  👤 Patient : {name}')
            print(f'  🕐 Date    : {dt_label}')
            print(f'  ⚤  Gender  : {w_gender.value.capitalize()}')
            print(f'  📁 Saved   : {img_path}')
            print('═' * 50)
            print('✅ Report ready! Click the Download button.')

    def on_download(b):
        if last_report['downloaded']:
            return
        if last_report['path']:
            last_report['downloaded'] = True
            btn_download.disabled     = True
            btn_download.description  = '✅ Downloaded'
            from google.colab import files
            files.download(last_report['path'])

    btn_generate.on_click(on_predict)
    btn_download.on_click(on_download)

    display(widgets.VBox([
        widgets.HTML(
            '<div style="background:#12152a;border-radius:8px;padding:14px 18px 10px;'
            'border-left:4px solid #74b9ff;margin-bottom:10px">'
            '<h3 style="color:#74b9ff;margin:0 0 4px;font-family:sans-serif">'
            '🏥 Symptom-Based Disease Risk Predictor</h3>'
            '<p style="color:#adb5bd;margin:0;font-size:12px">'
            'No lab tests needed — just describe your symptoms across all 7 tabs.</p></div>'
        ),
        tab,
        widgets.HTML('<div style="margin-top:8px"></div>'),
        btn_generate,
        btn_download,
        output_area,
    ]))
    print('✅ Symptom widget ready!')

except Exception as e:
    import traceback
    traceback.print_exc()
    print(f'❌ Widget error: {e}')


## 📥 CELL 14 — Download All Models & Reports

In [ ]:
# ══════════════════════════════════════════════════════════
# CELL 14 ▸ Package & Download Everything
# ══════════════════════════════════════════════════════════

from google.colab import files
import zipfile, os

zip_name = 'gender_aware_prediction_system.zip'
total_files = 0

with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in sorted(os.listdir('models')):
        zf.write(f'models/{fname}')
        print(f'  📦 models/{fname}')
        total_files += 1
    for fname in sorted(os.listdir('reports')):
        zf.write(f'reports/{fname}')
        print(f'  📊 reports/{fname}')
        total_files += 1

zip_size = os.path.getsize(zip_name) // 1024
print(f'\n✅ Packaged {total_files} files → {zip_name} ({zip_size} KB)')
files.download(zip_name)
print('⬇️  Download started!')

---
## ✅ Project Summary

| Component | Details |
|---|---|
| 🤖 Models | 9 × Random Forest (CalibratedClassifierCV) |
| ⚤ Gender Engine | Weighted risk prioritization (female/male) |
| 🛡️ Imbalance | SMOTE (if available) + `class_weight='balanced'` |
| 📊 Probabilities | Calibrated via Platt sigmoid scaling |
| 📈 Metrics | Accuracy + AUC-ROC + Confusion Matrix + ROC curves |
| 💾 Output | 9 `.pkl` models + PNG visual reports |
| 🎛️ UI | Interactive ipywidgets form |

### Dataset → Target Column Mapping (verified)
| Dataset | Target Column | Notes |
|---|---|---|
| breast_cancer_cleaned.csv | `diagnosis` | Already binary (0/1) |
| diabetes_cleaned.csv | `Outcome` | Binary (0/1) |
| heart_cleaned.xlsx | `target` | Binary (0/1) |
| kidney_cleaned_dataset.csv | `Target` | 5-class → binarized (0=healthy) |
| liver_cleaned.csv | `Dataset` | Binary (0/1) |
| lung_cleaned.csv | `Recovered` | Binary (0/1) |
| pcos_cleaned_dataset.csv | `PCOS (Y/N)` | Binary (0/1), typos auto-fixed |
| prostate_cleaned_dataset.csv | `lpsa` | Continuous → binarized at median |
| thyroid_cleaned_dataset.csv | `Recurred_Yes` | Boolean → int |

---
**🏆 Built for Hackathon — Gender-Aware Early Disease Risk Prediction System**